In [ ]:
# Install required libraries if not already installed
!pip install -q kaggle tensorflow matplotlib seaborn opencv-python scikit-learn keras-tuner

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("TensorFlow Version:", tf.__version__)

# Setup Kaggle API (Upload kaggle.json when prompted or ensure it's configured)
from google.colab import files
print("Please upload your kaggle.json file if running in Google Colab:")
files.upload() # Skip this if kaggle.json is already in ~/.kaggle/

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download Dataset
!kaggle datasets download -d pranavraikokte/covid19-image-dataset
!unzip -q covid19-image-dataset.zip -d dataset/
print("Dataset downloaded and extracted successfully!")

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Define dataset paths
base_dir = 'dataset/Covid19-dataset'
train_dir = os.path.join(base_dir, 'Train')
test_dir = os.path.join(base_dir, 'Test')

IMG_SIZE = 128
categories = ['Normal', 'Covid', 'Viral Pneumonia']

def load_data(data_dir):
    data = []
    labels = []
    for category in categories:
        path = os.path.join(data_dir, category)
        class_num = categories.index(category)
        if not os.path.exists(path):
            continue
        for img_name in os.listdir(path):
            try:
                img_path = os.path.join(path, img_name)
                img = cv2.imread(img_path)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                data.append(img)
                labels.append(class_num)
            except Exception as e:
                pass
    return np.array(data), np.array(labels)

print("Loading Training Data...")
X_train, y_train = load_data(train_dir)
print("Loading Test Data...")
X_test, y_test = load_data(test_dir)

print(f"Training Data Shape: {X_train.shape}, Labels Shape: {y_train.shape}")
print(f"Test Data Shape: {X_test.shape}, Labels Shape: {y_test.shape}")

# Display sample images from each class
plt.figure(figsize=(10, 3))
for i, category in enumerate(categories):
    idx = np.where(y_train == i)[0][0]
    plt.subplot(1, 3, i+1)
    plt.imshow(cv2.cvtColor(X_train[idx], cv2.COLOR_BGR2RGB))
    plt.title(f"Class: {category}")
    plt.axis('off')
plt.show()

In [ ]:
# Task 2: Normalize pixel values (0-255 -> 0.0-1.0)
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Further split Train into Train and Validation sets (80-20)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# Convert labels to categorical (one-hot encoding)
y_tr_cat = tf.keras.utils.to_categorical(y_tr, num_classes=3)
y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes=3)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=3)

# Task 3: EDA - Visualize Class Distribution
import seaborn as sns
sns.countplot(x=y_train)
plt.title("Class Distribution in Training Set")
plt.xticks(ticks=[0, 1, 2], labels=categories)
plt.xlabel("Classes")
plt.ylabel("Number of Images")
plt.show()

In [ ]:
# Model 1: Basic CNN
model_basic = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model_basic.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("Training Basic CNN...")
history_basic = model_basic.fit(X_tr, y_tr_cat, validation_data=(X_val, y_val_cat), epochs=10, batch_size=32)

# Model 2: Transfer Learning (ResNet50)
from tensorflow.keras.applications import ResNet50
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # Freeze base layers

model_resnet = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')
])

model_resnet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("Training ResNet50 Transfer Learning Model...")
history_resnet = model_resnet.fit(X_tr, y_tr_cat, validation_data=(X_val, y_val_cat), epochs=5, batch_size=32)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Evaluate Basic CNN
y_pred_basic = np.argmax(model_basic.predict(X_test), axis=1)
acc_basic = accuracy_score(y_test, y_pred_basic)
f1_basic = f1_score(y_test, y_pred_basic, average='weighted')

# Evaluate ResNet50
y_pred_resnet = np.argmax(model_resnet.predict(X_test), axis=1)
acc_resnet = accuracy_score(y_test, y_pred_resnet)
f1_resnet = f1_score(y_test, y_pred_resnet, average='weighted')

print(f"Basic CNN -> Test Accuracy: {acc_basic:.4f}, F1-Score: {f1_basic:.4f}")
print(f"ResNet50  -> Test Accuracy: {acc_resnet:.4f}, F1-Score: {f1_resnet:.4f}")

# Save your best model for deployment
model_resnet.save('covid_resnet_model.h5')
print("Best model saved successfully as 'covid_resnet_model.h5'!")